# **Notebook**: Landslide Exposure Database (LED) - Part 3

__Date__: September, 2025  
__Author__: D. Acosta-Reyes  
__Supervisor__: J. Wartman  
__University of Washington__

__Content__: Code utilized to aggregate information at the building-level  

See previous notebooks for:
* PART 1: Building Inventory Integration
    * Spatial Join and Nearest-Neighbor Comparison
    * Geometric filters
    * Proximity evaluation
    * Machine Learning Classification
    * Landslide Susceptibility Attribution  

* PART 2: Geographic Classification
    * Physiographic Divisions
    * Census Divisions
    * States
    * Census Tracts
    * Census Blocks  

__This notebook contains__:
* PART 3: Socioeconomic Attributes
    * Population
    * CRE
    * Poverty and Income
    * Error Metrics

### Initial setup

In [ ]:
'''Load and import necessary libraries'''
# General data libraries
import numpy as np
import pandas as pd
import geopandas as gpd

import os

# Define data path
data_path = "/Users/danielacosta/Library/CloudStorage/OneDrive-UW/0 - DA General Exam/Paper 1 - Exposure analysis/Data"

#### Load datasets for geographic classification

In [ ]:
# Load final inventories as data points (footprints)
building_inventory = gpd.read_file(os.path.join(data_path, "process_building_inventory", "final_overture_inventory_points_wGeo.gpkg"))
print(f'Building inventory records: {len(building_inventory)}')

nsi_matched_points = gpd.read_file(os.path.join(data_path, "process_building_inventory", "nsi_matched_points.gpkg"))
print(f'NSI matched points: {len(nsi_matched_points)}')

us_nsi = gpd.read_file(os.path.join(data_path, "nsi_buildings", "US_nsi_2022.gpkg"))
print(f'US NSI records: {len(us_nsi)}')

# load cenus blocks
census_blocks = gpd.read_file(os.path.join(data_path, "census_gov_data/tabblock20", "tl_2025_02_tabblock20.zip"))
print(f'Census blocks: {len(census_blocks)}')

# load census tracts
census_tracts = gpd.read_file(os.path.join(data_path, "census_gov_data", "cb_2024_us_tract_500k.zip"))
print(f'Census tracts: {len(census_tracts)}')

# load CRE data
cre_data = pd.read_csv(os.path.join(data_path, "census_gov_data", "CRE_23_Tract.csv"), encoding="ISO-8859-1")
print(f'CRE data records: {len(cre_data)}')

# load ACS data
acs_data_poverty = pd.read_csv(os.path.join(data_path, "census_gov_data", "ACS_2022_5YR_TRACT/extracted_tables/ACS_2022_X17_POVERTY_combined.csv"), encoding="ISO-8859-1")
acs_data_income = pd.read_csv(os.path.join(data_path, "census_gov_data", "ACS_2022_5YR_TRACT/extracted_tables/ACS_2022_X19_INCOME_combined.csv"), encoding="ISO-8859-1")
print(f'ACS poverty data records: {len(acs_data_poverty)}')
print(f'ACS income data records: {len(acs_data_income)}')

# load FFIEC data
ffiec_data = pd.read_csv(os.path.join(data_path, "census_gov_data", "FFIEC_Census_Tract_List_2023.csv"), encoding="ISO-8859-1")
print(f'FFIEC data records: {len(ffiec_data)}')

### A) Population Estimates at the building-level

In [ ]:
# use nsi_matched_points to update building_inventory with NSI matches
'''
nsi_source has three values: 
- nsi_loc (for direct link to NSI)
- nsi_near (for points that are near 10 m of an NSI match but not directly linked)
- nsi_avg (for points with no NSI match)
'''
# merge geoid to us_nsi to get state identifier for nsi_matched_points
us_nsi = us_nsi.merge(nsi_matched_points[['geoid', 'bid']], on='bid', how='left')

# Add NSI attributes to building_inventory based on nsi_matched_points
us_nsi['pop_day'] = us_nsi['pop2amu65'] + us_nsi['pop2amo65']
us_nsi['pop_night'] = us_nsi['pop2pmu65'] + us_nsi['pop2pmo65']
nsi_columns = ['st_damcat', 'num_story', 'sqft', 'med_yr_blt', 'val_struct', 'pop_day', 'pop_night']

# merge nsi_columns to building_inventory based on geoid
# nsi_loc and nsi_near. st_damcat take first, num_story take mean, sqft take sum, med_yr_blt take mean, val_struct take sum, pop_day take sum, pop_night take sum
nsi_to_merge = us_nsi[nsi_columns].groupby('geoid').agg(
    st_damcat=('st_damcat', 'first'),
    num_story=('num_story', 'mean'),
    sqft=('sqft', 'sum'),
    med_yr_blt=('med_yr_blt', 'mean'),
    val_struct=('val_struct', 'sum'),
    pop_day=('pop_day', 'sum'),
    pop_night=('pop_night', 'sum')
).reset_index()

building_inventory = building_inventory.merge(nsi_to_merge, on='geoid', how='left')
print(f'Building inventory with NSI attributes: {len(building_inventory)} records')

building_inventory_no_nsi = building_inventory[building_inventory['pop_day'].isna()].copy()
building_inventory_no_nsi.drop(columns=nsi_columns, inplace=True)

print(f'Building inventory without NSI attributes: {len(building_inventory_no_nsi)} records')

# Part 2: Use census blocks to fill in missing NSI attributes for building_inventory points that do not have NSI matches (nsi_loc, nsi_near) but are within census blocks with NSI matches (nsi_avg)
# First, spatial join census blocks to us_nsi to get GEOID20
if us_nsi.crs != census_blocks.crs:
    us_nsi = us_nsi.to_crs(census_blocks.crs)

us_nsi_with_block = gpd.sjoin(us_nsi, census_blocks[['GEOID20', 'geometry']], how='left', predicate='intersects')

# get metrics for census blocks with NSI all
nsi_block_metrics = us_nsi_with_block.groupby('GEOID20').agg(
    st_damcat=('st_damcat', lambda x: x.mode().iloc[0] if not x.mode().empty else np.nan),
    num_story=('num_story', 'mean'),
    med_yr_blt=('med_yr_blt', 'mean'),
    val_struct=('val_struct', 'mean'),
    pop_day=('pop_day', 'sum'),
    pop_night=('pop_night', 'sum')
).reset_index()

# merge nsi_block_metrics back to census_blocks to get NSI metrics for all census blocks
census_blocks = census_blocks.merge(nsi_block_metrics, on='GEOID20', how='left')
    

#### Assign Population based on weighted average

In [ ]:
def assign_building_population(source_df, target_df, census_df=None):
    """
    Assigns daytime and nighttime population to buildings based on 
    sqft density derived from a reference spatial dataset (e.g., NSI).
    
    Parameters:
    - source_df: DataFrame with ground-truth NSI matches (must contain 'st_damcat', 'sqft', 'pop_day', 'pop_night', 'GEOID20')
    - target_df: DataFrame of buildings needing population (must contain 'footprint_area_sqm', 'GEOID20')
    - census_df: (Optional) DataFrame with actual census population for correction (must contain 'GEOID20', 'POP20')
    
    Returns:
    - Target DataFrame with 'estimated_pop_day' and 'estimated_pop_night' assigned.
    """
    print("Starting population density assignment...")
    
    # ---- 1. Filter Source Data to Residential ----
    # Assuming residential structures are the primary drivers of base population
    res_source = source_df[source_df['st_damcat'] == 'RES'].copy()
    res_source['STATEFP'] = res_source['GEOID20'].astype(str).str[:2]
    
    # ---- 2. Calculate Densities at 3 Geographic Tiers ----
    # Tier 1: Census Block Level
    block_agg = res_source.groupby('GEOID20')[['sqft', 'pop_day', 'pop_night']].sum()
    block_dens_day = (block_agg['pop_day'] / block_agg['sqft']).replace([np.inf, -np.inf], np.nan)
    block_dens_night = (block_agg['pop_night'] / block_agg['sqft']).replace([np.inf, -np.inf], np.nan)
    
    # Tier 2: State Level
    state_agg = res_source.groupby('STATEFP')[['sqft', 'pop_day', 'pop_night']].sum()
    state_dens_day = (state_agg['pop_day'] / state_agg['sqft']).replace([np.inf, -np.inf], np.nan)
    state_dens_night = (state_agg['pop_night'] / state_agg['sqft']).replace([np.inf, -np.inf], np.nan)
    
    # Tier 3: National Level (Global Fallback)
    nat_dens_day = res_source['pop_day'].sum() / res_source['sqft'].sum()
    nat_dens_night = res_source['pop_night'].sum() / res_source['sqft'].sum()

    # ---- 3. Prepare Target Data ----
    target = target_df.copy()
    target['GEOID20'] = target['GEOID20'].astype(str)
    target['STATEFP'] = target['GEOID20'].str[:2]
    target['sqft'] = np.round(target['footprint_area_sqm'] * 10.7639)
    
    # ---- 4. Cascade Assignment (Block -> State -> National) ----
    # Map block density first
    target['pop_day_density'] = target['GEOID20'].map(block_dens_day)
    target['pop_night_density'] = target['GEOID20'].map(block_dens_night)
    
    # Fill missing with State density
    target['pop_day_density'] = target['pop_day_density'].fillna(target['STATEFP'].map(state_dens_day))
    target['pop_night_density'] = target['pop_night_density'].fillna(target['STATEFP'].map(state_dens_night))
    
    # Fill remaining missing with National density
    target['pop_day_density'] = target['pop_day_density'].fillna(nat_dens_day)
    target['pop_night_density'] = target['pop_night_density'].fillna(nat_dens_night)
    
    print(f"Density successfully assigned to {target['pop_day_density'].notnull().sum()} of {len(target)} records.")

    # ---- 5. Calculate Initial Estimates ----
    target['estimated_pop_day'] = target['pop_day_density'] * target['sqft']
    target['estimated_pop_night'] = target['pop_night_density'] * target['sqft']

    # ---- 6. Apply Census Correction Factor (If Provided) ----
    if census_df is not None:
        print("Applying Census Block correction factors...")
        # Get total estimated night population per block
        block_est = target.groupby('GEOID20')['estimated_pop_night'].sum().reset_index(name='total_est_night')
        
        # Merge with actual census population
        correction_df = block_est.merge(census_df[['GEOID20', 'POP20']], on='GEOID20', how='left')
        
        # Calculate factor: (Actual POP20) / (Our Estimated Pop)
        # Cap lower bound at 0.51 to avoid dropping population to near zero
        correction_df['correction_factor'] = np.maximum(0.51, correction_df['POP20'] / correction_df['total_est_night'].replace(0, np.nan))
        correction_df['correction_factor'] = correction_df['correction_factor'].replace([np.inf, -np.inf], np.nan).fillna(1.0)
        
        # Map factor back to target buildings and adjust
        factor_map = dict(zip(correction_df['GEOID20'], correction_df['correction_factor']))
        target['correction_factor'] = target['GEOID20'].map(factor_map).fillna(1.0)
        
        target['pop_day'] = target['estimated_pop_day'] * target['correction_factor']
        target['pop_night'] = target['estimated_pop_night'] * target['correction_factor']

    # ---- 7. Final Formatting (Floor at 1, Round to Integer) ----
    target['pop_day'] = target['pop_day'].apply(lambda x: max(1, round(x)) if pd.notnull(x) else 1)
    target['pop_night'] = target['pop_night'].apply(lambda x: max(1, round(x)) if pd.notnull(x) else 1)
    
    # Cleanup intermediate columns if desired
    cols_to_drop = ['STATEFP', 'pop_day_density', 'pop_night_density', 'correction_factor']
    target = target.drop(columns=[c for c in cols_to_drop if c in target.columns])

    print(f"Final Total Est Day Pop: {target['pop_day'].sum():,.0f}")
    print(f"Final Total Est Night Pop: {target['pop_night'].sum():,.0f}")
    
    return target

In [ ]:
# Run the population assignment function
building_inventory_avg = assign_building_population(
    source_df=us_nsi_with_block, 
    target_df=building_inventory_no_nsi,
    census_df=census_blocks  # Requires columns: 'GEOID20', 'POP20'
)

# merge NSI attributes from nsi_block_metrics to building_inventory_avg based on GEOID20
building_inventory_avg = building_inventory_avg.merge(nsi_block_metrics[['GEOID20', 'st_damcat', 'num_story', 'med_yr_blt', 'val_struct']], on='GEOID20', how='left')

# concatenate building_inventory nan with building_inventory_avg to get final building inventory with population estimates
building_inventory_pop = pd.concat([building_inventory[building_inventory['pop_day'].notna()].copy(), building_inventory_avg], ignore_index=True)
print(f'Final building inventory with population estimates: {len(building_inventory_pop)} records')

### B) Error metrics in Population Estimates

In [ ]:
import pandas as pd
import numpy as np

# 1. Load datasets 
df_buildings = building_inventory_pop[['GEOID20', 'pop_night', 'susc_class','UR20']].copy()                                   
df_blocks = census_blocks[['GEOID20', 'POP20', 'pop_night']].copy()
# rename pop_night in df_blocks to pop_night_nsi to avoid confusion
df_blocks = df_blocks.rename(columns={'pop_night': 'pop_night_nsi'})

# reduction factor to NSI estimaes
national_ref = df_blocks['pop_night_nsi'].sum()  # Total population from NSI estimates at block level
reduction_factor =  national_ref / df_buildings['pop_night'].sum()  # To scale down to 331.4M for error calculation
df_buildings['pop_night_reduced'] = df_buildings['pop_night'] * reduction_factor

def calculate_spatial_error(df_buildings, df_blocks):
    # Step A: Aggregate building population to block level
    # This is the "Estimated" population per block
    print("Aggregating building-level data...")
    block_est = df_buildings.groupby('GEOID20')['pop_night_reduced'].sum().reset_index()
    block_est.columns = ['GEOID20', 'pop_estimated']

    # Step B: Merge with Census Truth (331.4M baseline)
    print("Merging with census ground truth...")
    comparison = pd.merge(block_est, df_blocks, on='GEOID20', how='outer').fillna(0)
    # Step C: Calculate Residuals
    # Residual = (What we estimated) - (What Census says)
    comparison['residual'] = comparison['pop_estimated'] - comparison['POP20']
    
    # Step D: Calculate Global Metrics
    n = len(comparison)
    residuals = comparison['residual']
    
    rmse = np.sqrt(np.mean(residuals**2))
    mae = np.mean(np.abs(residuals))
    
    # Step E: Calculate the 95% Confidence Interval (CI)
    # This represents the uncertainty in the spatial distribution
    std_error = np.std(residuals)
    margin_of_error_95 = 1.96 * (std_error / np.sqrt(n))
    
    # Calculate Mean Absolute Percentage Error (MAPE) for relative error
    # We filter out zero-pop blocks to avoid division by zero
    mask = comparison['POP20'] > 0
    mape = np.mean(np.abs(comparison.loc[mask, 'residual'] / comparison.loc[mask, 'POP20'])) * 100

    print(f"--- Global Error Results ---")
    print(f"Total Blocks Analyzed: {n:,}")
    print(f"RMSE: {rmse:.4f} people per block")
    print(f"MAE: {mae:.4f} people per block")
    print(f"MAPE: {mape:.2f}%")
    print(f"95% CI Margin of Error: ±{margin_of_error_95:.6f}")
    
    return comparison, rmse, mape

# Run the spatial error calculation
comparison, rmse, mape = calculate_spatial_error(df_buildings, df_blocks)

# To scale your result to the 342.9M Target after validation:
# comparison['pop_final_2025'] = comparison['pop_estimated'] * (342.9 / 331.4)

In [ ]:
# Rural and Urban Analysis

def analyze_residuals_by_UR20(df_comparison):

    # Merge UR20 classification
    ur20_data = census_blocks[['GEOID20', 'UR20']].copy()
    df_comparison = df_comparison.merge(ur20_data, on='GEOID20', how='left')
    
    # 1. Group by the official UR20 classification
    summary = df_comparison.groupby('UR20').agg(
        total_blocks=('GEOID20', 'count'),
        total_census_pop=('POP20', 'sum'),
        total_est_pop=('pop_estimated', 'sum'),
        mean_residual=('residual', 'mean'),
        # Standard Deviation of the error
        std_residual=('residual', 'std'),
        # Root Mean Square Error
        rmse=('residual', lambda x: np.sqrt(np.mean(x**2))),
        # Mean Absolute Error
        mae=('residual', lambda x: np.mean(np.abs(x)))
    ).reset_index()

    # 2. Calculate Normalized RMSE (nRMSE)
    # We divide RMSE by the average population per block in that setting
    # This shows the error RELATIVE to the size of the population in those blocks
    summary['avg_block_pop'] = summary['total_census_pop'] / summary['total_blocks']
    summary['nRMSE'] = summary['rmse'] / summary['avg_block_pop']

    # 3. Calculate MAPE per setting
    # Excluding zero-pop blocks to avoid infinity
    mask = df_comparison['POP20'] > 0
    mape_u = np.mean(np.abs(df_comparison[(mask) & (df_comparison['UR20'] == 'U')]['residual'] / 
                           df_comparison[(mask) & (df_comparison['UR20'] == 'U')]['POP20'])) * 100
    mape_r = np.mean(np.abs(df_comparison[(mask) & (df_comparison['UR20'] == 'R')]['residual'] / 
                           df_comparison[(mask) & (df_comparison['UR20'] == 'R')]['POP20'])) * 100
    print("--- Urban (U) vs. Rural (R) Error Summary ---")
    print(summary)
    print(f"\nMAPE Urban: {mape_u:.2f}%")
    print(f"MAPE Rural: {mape_r:.2f}%")
    
    return summary

# Run the Urban vs Rural analysis
summary_urban_rural = analyze_residuals_by_UR20(comparison)


### C) Community Resilience Estimate (CRE)

In [ ]:
# Merge CRE data with building_inventory_pop by GEOIDFQ on building_inventory_pop and GEO_ID on CRE data
cre_data = cre_data.rename(columns={'GEO_ID': 'GEOIDFQ'})
building_inventory_pop = building_inventory_pop.merge(cre_data[['GEOIDFQ','PRED0_PE','PRED12_PE', 'PRED3_PE']], on='GEOIDFQ', how='left')
print(building_inventory_pop[['GEOIDFQ','PRED0_PE','PRED12_PE', 'PRED3_PE']].head(3))

### D) ACS: Poverty and Income Data

In [ ]:
# Merge ACS poverty data with building_inventory_pop by GEOIDFQ on building_inventory_pop and GEO_ID on ACS data
building_inventory_pop = building_inventory_pop.merge(acs_data_poverty[['GEOIDFQ','B17017_E002','B17017_E001']], on='GEOIDFQ', how='left')

# calculate poverty rate
building_inventory_pop['POVERTY_RATE'] = building_inventory_pop['B17017_E002'] / building_inventory_pop['B17017_E001']
print(building_inventory_pop[['GEOIDFQ','B17017_E002','B17017_E001','POVERTY_RATE']].head(3))

In [ ]:
# merge ACS income data with building_inventory_pop by GEOIDFQ on building_inventory_pop and GEO_ID on ACS data
building_inventory_pop = building_inventory_pop.merge(acs_data_income[['GEOIDFQ','B19013_E001']], on='GEOIDFQ', how='left')
print(building_inventory_pop[['GEOIDFQ','B19013_E001']].head(3))

### E) FFIEC Income Level Classification

In [ ]:
# Merge Tract Income Level with building_inventory_pop by GEOIDFQ[:11] on building_inventory_pop and FIPS code on ffiec_data
# First, we need to extract the FIPS code from GEOIDFQ in building_inventory_pop (which is the first 11 characters)
building_inventory_pop['FIPS_code'] = building_inventory_pop['GEOIDFQ'].str[:11]
# FIPS code in ffiec_data is integer, so we need to convert it to string and pad with zeros to 11 characters
ffiec_data['FIPS code'] = ffiec_data['FIPS code'].astype(str).str.zfill(11)
# rename FIPS code in ffiec data to match building_inventory_pop
ffiec_data = ffiec_data.rename(columns={'FIPS code': 'FIPS_code'})

building_inventory_pop = building_inventory_pop.merge(ffiec_data[['FIPS_code', 'Tract income level']], on='FIPS_code', how='left')
# rename Tract income level to tract_income_level
building_inventory_pop = building_inventory_pop.rename(columns={'Tract income level': 'Tract_income_level'})
print(building_inventory_pop[['FIPS_code', 'Tract_income_level']].head(3))

#### Landslide Exposure Database Completed! Save to data_path

In [ ]:
building_inventory_pop.to_file(os.path.join(data_path, "process_building_inventory", "US_LandslideExposureDatabase_points.gpkg"), driver='GPKG')

#### End of Script

#